Sebastian Holm Bangsø - ljp445

In [3]:
#2.1
import pandas as pd

df = pd.read_stata('A1_kommune.dta')

overblik = df[['taxrev', 'taxrate', 'pop']].describe()
print(overblik)

             taxrev    taxrate            pop
count     98.000000  98.000000      98.000000
mean    4477.341309  25.208162   56475.887755
std     5251.175293   0.908003   62925.301713
min      211.228409  22.799999    1969.000000
25%     2466.702271  24.799999   29997.750000
50%     3317.848633  25.299999   43475.000000
75%     4786.060913  25.700001   59733.000000
max    44170.335938  27.799999  528208.000000


In [4]:
import numpy as np
import pandas as pd
import scipy.stats as stats
df['log_taxrev'] = np.log(df['taxrev'])
df['log_pop'] = np.log(df['pop'])
df['const'] = 1


def OLS_SLR(x, y):
    n = len(x)
    x_bar = np.mean(x)
    y_bar = np.mean(y)

    # Estimater
    cov_xy = np.sum((x - x_bar) * (y - y_bar))
    var_x  = np.sum((x - x_bar) ** 2)
    beta1hat = cov_xy / var_x
    beta0hat = y_bar - (beta1hat * x_bar)
    
    # Residualer og R^2
    y_hat = beta0hat + (beta1hat * x)
    u_hat = y - y_hat
    SSE = np.sum((y_hat - y_bar)**2)
    SST = np.sum((y - y_bar)**2)
    SSR = np.sum(u_hat**2)
    R2 = SSE / SST

    # Standardfejl (Ændret fra 'df' til 'f_grader', så den ikke conflict'er med pandas/df)
    f_grader = n - 2
    resvar = SSR / f_grader
    var_beta1hat = resvar / var_x
    var_beta0hat = (resvar * np.sum(x**2) / n) / var_x
    se_beta1hat = np.sqrt(var_beta1hat)
    se_beta0hat = np.sqrt(var_beta0hat)

    # T-statistikker
    t_beta1 = beta1hat / se_beta1hat
    t_beta0 = beta0hat / se_beta0hat

    p_beta1 = 2 * stats.t.sf(np.abs(t_beta1), f_grader)
    p_beta0 = 2 * stats.t.sf(np.abs(t_beta0), f_grader)

    # Tabel layout
    results = pd.DataFrame({
        'Koefficient': [beta0hat, beta1hat],
        'Std.fejl': [se_beta0hat, se_beta1hat],
        't-værdi': [t_beta0, t_beta1],
        'P>|t|': [p_beta0, p_beta1]
    }, index=['Konstant (beta0)', 'Skatteprocent (beta1)'])
    
    results.loc['R-squared'] = [R2, np.nan, np.nan, np.nan]

    formatted_results = results.round(4).fillna('')
    
    display(formatted_results)

OLS_SLR(df['taxrate'], df['log_taxrev'])

,Koefficient,Std.fejl,t-værdi,P>|t|
Konstant (beta0),11.6982,2.143,5.4587,0.0
Skatteprocent (beta1),-0.1426,0.085,-1.6787,0.0965
R-squared,0.0285,,,


In [5]:
#2.5
import numpy as np
import pandas as pd
import scipy.stats as stats

def OLS_ren(X, y, navne):

    n = X.shape[0]
    k = X.shape[1] - 1


    X_T_X_inv = np.linalg.inv(X.T @ X)
    betahat = X_T_X_inv @ X.T @ y
    
    y_hat = X @ betahat
    u_hat = y - y_hat
    y_bar = np.mean(y)

    SST = np.sum((y - y_bar)**2)
    SSR = np.sum(u_hat**2)
    SSE = np.sum((y_hat - y_bar)**2)
    R2 = SSE / SST
    
    resvar = (1 / (n - k - 1)) * np.dot(u_hat, u_hat)
    
    var_betahatX = resvar * X_T_X_inv
    se_betahat = np.sqrt(np.diagonal(var_betahatX))
    
    t_stat = betahat / se_betahat
    p_values = 2 * stats.t.sf(np.abs(t_stat), n - k - 1)
    
    #tabel layout
    results = pd.DataFrame({
        'Koefficient': betahat,
        'Std.fejl': se_betahat,
        't-værdi': t_stat,
        'P>|t|': p_values
    }, index=navne)
    
    results.loc['R-squared'] = [R2, np.nan, np.nan, np.nan]
    formatted_results = results.round(4).fillna('')
    
    display(formatted_results)



df['konstant'] = 1


data_kolonner = ['konstant', 'taxrate', 'log_pop']
pæne_navne = ['Konstant (beta0)', 'Skatteprocent (beta1)', 'Befolkningstal (beta2)']

X_tal = df[data_kolonner].values
y_tal = df['log_taxrev'].values


OLS_ren(X_tal, y_tal, pæne_navne)

,Koefficient,Std.fejl,t-værdi,P>|t|
Konstant (beta0),-2.8022,0.3756,-7.461,0.0
Skatteprocent (beta1),0.0226,0.0125,1.8165,0.0725
Befolkningstal (beta2),0.9711,0.0144,67.4707,0.0
R-squared,0.9801,,,


In [6]:
#3.3 del 1
import statsmodels.api as sm
df['log_taxrev'] = np.log(df['taxrev'])
df['log_pop'] = np.log(df['pop'])
df['const'] = 1

results = sm.OLS(df['taxrate'], df[['const', 'log_pop']]).fit()

df['res1'] = results.resid
step2_model = sm.OLS(df['log_taxrev'], df[['const', 'res1']]).fit()


print(results.summary())

                            OLS Regression Results                            
Dep. Variable:                taxrate   R-squared:                       0.039
Model:                            OLS   Adj. R-squared:                  0.029
Method:                 Least Squares   F-statistic:                     3.862
Date:                Thu, 17 Sep 2026   Prob (F-statistic):             0.0523
Time:                        18:25:59   Log-Likelihood:                -127.16
No. Observations:                  98   AIC:                             258.3
Df Residuals:                      96   BIC:                             263.5
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         27.6268      1.234     22.386      0.0

In [7]:
#3,3 del 2
print(step2_model.summary())

                            OLS Regression Results                            
Dep. Variable:             log_taxrev   R-squared:                       0.001
Model:                            OLS   Adj. R-squared:                 -0.010
Method:                 Least Squares   F-statistic:                   0.06626
Date:                Thu, 17 Sep 2026   Prob (F-statistic):              0.797
Time:                        18:25:59   Log-Likelihood:                -112.50
No. Observations:                  98   AIC:                             229.0
Df Residuals:                      96   BIC:                             234.2
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          8.1031      0.078    104.100      0.0